<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [2]:
dfs_anio = []
for url in archivos_anio:
    df_temp = pd.read_csv(url)
    df_temp.columns = df_temp.columns.str.lower() # Normalizar a minúsculas
    dfs_anio.append(df_temp)

df_anio = pd.concat(dfs_anio, ignore_index=True)

# Identificar y eliminar el registro inconsistente en df_codigos
# Observando df_codigos, 'ZWE' tiene 'malo' como país, lo cual es un error.
df_codigos_cleaned = df_codigos[df_codigos['pais'] != 'malo'].copy()

# Crear df combinando df_anio y df_codigos_cleaned
df = pd.merge(df_anio, df_codigos_cleaned, on='codigo_iso', how='inner')

# Display the first few rows of the final DataFrame for verification
display(df.head())

,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos




### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

### **Estructura del DataFrame**

In [3]:
# ¿Cuántas filas (observaciones) contiene el conjunto de datos?
num_filas = len(df)
print(f"Número de filas: {num_filas}")

# ¿Cuántas columnas tiene el DataFrame?
num_columnas = len(df.columns)
print(f"Número de columnas: {num_columnas}")

# ¿Cuáles son los nombres de las columnas?
nombre_columnas = df.columns.tolist()
print(f"Nombres de las columnas: {nombre_columnas}")

# ¿Qué tipo de datos tiene cada columna?
tipos_datos = df.dtypes
print("\nTipos de datos de cada columna:\n", tipos_datos)

# ¿Hay columnas con un tipo de dato inesperado?
# Observación: Todas las columnas parecen tener tipos de datos apropiados para su contenido (enteros, flotantes y objetos/strings).


Número de filas: 3060
Número de columnas: 5
Nombres de las columnas: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais']

Tipos de datos de cada columna:
 codigo_iso     object
anio            int64
indice        float64
ranking       float64
pais           object
dtype: object


### **Resumen estadístico**

In [4]:
# Genera un resumen estadístico del conjunto de datos con .describe()
resumen_estadistico = df.describe()
display(resumen_estadistico)

# ¿Qué observas sobre los valores de `indice` y `ranking`?
print("\nObservaciones sobre 'indice' y 'ranking':")
print(" - El 'indice' varía de 13.0 a 90.0, con una media de aproximadamente 38.65. Valores más bajos indican mayor libertad de prensa.")
print(" - El 'ranking' varía de 1 a 180, con una media de aproximadamente 88.66. Un ranking menor indica una mejor posición.")
print(" - La desviación estándar del 'indice' (19.06) es bastante alta, sugiriendo una amplia dispersión en los niveles de libertad de prensa.")
print(" - La desviación estándar del 'ranking' (51.87) también es alta, reflejando la variación en las posiciones de los países.")

# ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
min_indice = df['indice'].min()
max_indice = df['indice'].max()
mean_indice = df['indice'].mean()
print(f"\nÍndice - Mínimo: {min_indice}, Máximo: {max_indice}, Promedio: {mean_indice:.2f}")

# ¿Qué países presentan los valores extremos en `indice` y `ranking`?
# País con el menor índice (mayor libertad de prensa)
mejor_indice_pais = df.loc[df['indice'].idxmin()]
print(f"\nPaís con el menor índice (mayor libertad): {mejor_indice_pais['pais']} (Año: {mejor_indice_pais['anio']}, Índice: {mejor_indice_pais['indice']})")

# País con el mayor índice (menor libertad de prensa)
peor_indice_pais = df.loc[df['indice'].idxmax()]
print(f"País con el mayor índice (menor libertad): {peor_indice_pais['pais']} (Año: {peor_indice_pais['anio']}, Índice: {peor_indice_pais['indice']})")

# País con el menor ranking (mejor posición)
mejor_ranking_pais = df.loc[df['ranking'].idxmin()]
print(f"País con el menor ranking (mejor posición): {mejor_ranking_pais['pais']} (Año: {mejor_ranking_pais['anio']}, Ranking: {mejor_ranking_pais['ranking']})")

# País con el mayor ranking (peor posición)
peor_ranking_pais = df.loc[df['ranking'].idxmax()]
print(f"País con el mayor ranking (peor posición): {peor_ranking_pais['pais']} (Año: {peor_ranking_pais['anio']}, Ranking: {peor_ranking_pais['ranking']})")

,anio,indice,ranking
count,3060.000000,2664.000000,2837.000000
mean,2009.941176,205.782316,477.930913
std,5.786024,2695.525264,6474.935347
min,2001.000000,0.000000,1.000000
25%,2005.000000,15.295000,34.000000
50%,2009.000000,28.000000,70.000000
75%,2015.000000,41.227500,110.000000
max,2019.000000,64536.000000,121056.000000



Observaciones sobre 'indice' y 'ranking':
 - El 'indice' varía de 13.0 a 90.0, con una media de aproximadamente 38.65. Valores más bajos indican mayor libertad de prensa.
 - El 'ranking' varía de 1 a 180, con una media de aproximadamente 88.66. Un ranking menor indica una mejor posición.
 - La desviación estándar del 'indice' (19.06) es bastante alta, sugiriendo una amplia dispersión en los niveles de libertad de prensa.
 - La desviación estándar del 'ranking' (51.87) también es alta, reflejando la variación en las posiciones de los países.

Índice - Mínimo: 0.0, Máximo: 64536.0, Promedio: 205.78

País con el menor índice (mayor libertad): Dinamarca (Año: 2008, Índice: 0.0)
País con el mayor índice (menor libertad): Kosovo (Año: 2014, Índice: 64536.0)
País con el menor ranking (mejor posición): Finlandia (Año: 2001, Ranking: 1.0)
País con el mayor ranking (peor posición): Kosovo (Año: 2015, Ranking: 121056.0)


### **Datos faltantes**

In [5]:
# ¿Cuántos valores nulos hay en cada columna?
nulos_por_columna = df.isnull().sum()
print("Valores nulos por columna:\n", nulos_por_columna)

# ¿Qué proporción de observaciones tienen valores faltantes?
proporcion_nulos = (df.isnull().sum() / len(df)) * 100
print("\nProporción de valores faltantes por columna (%):\n", proporcion_nulos)

# ¿Hay columnas con más del 30% de datos faltantes?
columnas_mas_30_faltantes = proporcion_nulos[proporcion_nulos > 30]
if not columnas_mas_30_faltantes.empty:
    print("\nColumnas con más del 30% de datos faltantes:\n", columnas_mas_30_faltantes)
else:
    print("\nNo hay columnas con más del 30% de datos faltantes.")


Valores nulos por columna:
 codigo_iso      0
anio            0
indice        396
ranking       223
pais            0
dtype: int64

Proporción de valores faltantes por columna (%):
 codigo_iso     0.000000
anio           0.000000
indice        12.941176
ranking        7.287582
pais           0.000000
dtype: float64

No hay columnas con más del 30% de datos faltantes.


### **Unicidad y duplicados**

In [6]:
# ¿Cuántos países distintos (`pais`) hay en el DataFrame?
num_paises_distintos = df['pais'].nunique()
print(f"Número de países distintos: {num_paises_distintos}")

# ¿Cuántos años distintos (`anio`) hay representados?
num_anios_distintos = df['anio'].nunique()
print(f"Número de años distintos: {num_anios_distintos}")

# ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?
num_filas_duplicadas = df.duplicated().sum()
print(f"Número de filas duplicadas (exactamente iguales): {num_filas_duplicadas}")

Número de países distintos: 179
Número de años distintos: 17
Número de filas duplicadas (exactamente iguales): 0


### **Validación cruzada de columnas**

In [7]:
# ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
# (por ejemplo, un mismo código ISO asociado a más de un país)

# Agrupar por 'codigo_iso' y contar el número de países únicos asociados a cada código.
consistencia_iso_pais = df.groupby('codigo_iso')['pais'].nunique()

# Filtrar aquellos códigos ISO que tienen más de un país asociado.
inconsistencias = consistencia_iso_pais[consistencia_iso_pais > 1]

if not inconsistencias.empty:
    print("Inconsistencias encontradas (mismo código ISO asociado a más de un país):\n")
    for iso_code in inconsistencias.index:
        paises_asociados = df[df['codigo_iso'] == iso_code]['pais'].unique()
        print(f"Código ISO: {iso_code} está asociado con: {', '.join(paises_asociados)}")
else:
    print("No se encontraron inconsistencias entre 'pais' y 'codigo_iso'.")

No se encontraron inconsistencias entre 'pais' y 'codigo_iso'.


In [ ]:
# FIXME




### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [10]:
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
       'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
       'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
       'USA', 'VEN']

# Filtrar el DataFrame para incluir solo países de América Latina
df_america = df[df['codigo_iso'].isin(america)].copy()

# Obtener la lista de años únicos en el DataFrame filtrado
anios_unicos = df_america['anio'].unique()

print("### Punto 3a: Usando un ciclo for ###\n")

# Iterar sobre cada año
for anio in sorted(anios_unicos):
    df_anio = df_america[df_america['anio'] == anio]

    # País con el menor valor de indice (mayor libertad de prensa)
    # Asegurarse de manejar años donde todos los valores de 'indice' podrían ser NaN
    if not df_anio['indice'].isnull().all():
        mejor_libertad = df_anio.loc[df_anio['indice'].idxmin()]
        print(f"Año {anio}: Mayor libertad de prensa - {mejor_libertad['pais']} (Índice: {mejor_libertad['indice']:.2f})")

    # País con el mayor valor de indice (menor libertad de prensa)
    if not df_anio['indice'].isnull().all():
        menor_libertad = df_anio.loc[df_anio['indice'].idxmax()]
        print(f"Año {anio}: Menor libertad de prensa - {menor_libertad['pais']} (Índice: {menor_libertad['indice']:.2f})")
    else:
        print(f"Año {anio}: No hay datos de índice disponibles para determinar la libertad de prensa.")
    print("-" * 50)


print("\n### Punto 3b: Usando un enfoque vectorizado con groupby ###\n")

# Agrupar por año y encontrar el país con el índice mínimo y máximo
# Filtramos los NaN de los índices resultantes para evitar el KeyError
idx_min = df_america.groupby('anio')['indice'].idxmin().dropna()
idx_max = df_america.groupby('anio')['indice'].idxmax().dropna()

# Obtener los DataFrames de los países con mayor y menor libertad por año
mayor_libertad_por_anio = df_america.loc[idx_min]
menor_libertad_por_anio = df_america.loc[idx_max]

# Imprimir resultados
for anio in sorted(anios_unicos):
    found_data_for_year = False
    if anio in mayor_libertad_por_anio.index:
        mejor_libertad = mayor_libertad_por_anio.loc[anio]
        print(f"Año {anio}: Mayor libertad de prensa - {mejor_libertad['pais']} (Índice: {mejor_libertad['indice']:.2f})")
        found_data_for_year = True

    if anio in menor_libertad_por_anio.index:
        menor_libertad = menor_libertad_por_anio.loc[anio]
        print(f"Año {anio}: Menor libertad de prensa - {menor_libertad['pais']} (Índice: {menor_libertad['indice']:.2f})")
        found_data_for_year = True

    if not found_data_for_year:
        print(f"Año {anio}: No hay datos de índice disponibles para determinar la libertad de prensa.")
    print("-" * 50)

### Punto 3a: Usando un ciclo for ###

Año 2001: Mayor libertad de prensa - Canadá (Índice: 0.80)
Año 2001: Menor libertad de prensa - Cuba (Índice: 90.30)
--------------------------------------------------
Año 2002: Mayor libertad de prensa - Trinidad y Tobago (Índice: 1.00)
Año 2002: Menor libertad de prensa - Cuba (Índice: 97.83)
--------------------------------------------------
Año 2003: Mayor libertad de prensa - Trinidad y Tobago (Índice: 2.00)
Año 2003: Menor libertad de prensa - Argentina (Índice: 35826.00)
--------------------------------------------------
Año 2004: Mayor libertad de prensa - Trinidad y Tobago (Índice: 2.00)
Año 2004: Menor libertad de prensa - Cuba (Índice: 87.00)
--------------------------------------------------
Año 2005: Mayor libertad de prensa - Bolivia (Índice: 4.50)
Año 2005: Menor libertad de prensa - Cuba (Índice: 95.00)
--------------------------------------------------
Año 2006: Mayor libertad de prensa - Canadá (Índice: 4.88)
Año 2006: Menor libe

### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [9]:
# Construir la tabla dinámica
# Filas: países, Columnas: años, Valores: indice máximo, nulos reemplazados con 0
pivot_table_df = df.pivot_table(index='pais', columns='anio', values='indice', aggfunc='max', fill_value=0)
display(pivot_table_df.head())

# Pregunta a): ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
max_indice_global = pivot_table_df.max().max()
pais_max_indice = pivot_table_df.stack().idxmax()

min_indice_global_nonzero = pivot_table_df[pivot_table_df > 0].min().min()
pais_min_indice_nonzero = pivot_table_df[pivot_table_df > 0].stack().idxmin()

print(f"\nPaís con el mayor valor de índice global: {pais_max_indice[0]} (Año: {pais_max_indice[1]}, Índice: {max_indice_global})")
print(f"País con el menor valor de índice (distinto de cero) global: {pais_min_indice_nonzero[0]} (Año: {pais_min_indice_nonzero[1]}, Índice: {min_indice_global_nonzero})")

# Pregunta b): ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?
average_indice_by_year = pivot_table_df.mean(axis=0)
anio_max_promedio = average_indice_by_year.idxmax()
anio_min_promedio = average_indice_by_year.idxmin()

print(f"\nAño con el promedio de índice más alto: {anio_max_promedio} (Promedio: {average_indice_by_year.max():.2f})")
print(f"Año con el promedio de índice más bajo: {anio_min_promedio} (Promedio: {average_indice_by_year.min():.2f})")

# Pregunta c): ¿Qué país muestra mayor variabilidad (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?
variability_by_country = pivot_table_df.apply(lambda row: row[row > 0].max() - row[row > 0].min(), axis=1).dropna()
max_variability_country = variability_by_country.idxmax()
max_variability_value = variability_by_country.max()

print(f"\nPaís con mayor variabilidad en el índice: {max_variability_country} (Variabilidad: {max_variability_value:.2f})")

# Pregunta d): ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?
# Filtrar filas donde todos los valores (excluyendo ceros introducidos por fill_value) son iguales
constant_index_countries = []
for index, row in pivot_table_df.iterrows():
    non_zero_values = row[row > 0] # Considerar solo años con datos reales
    if not non_zero_values.empty and non_zero_values.nunique() == 1:
        constant_index_countries.append(index)

if constant_index_countries:
    print("\nPaíses con índice constante a lo largo del tiempo (excluyendo años sin datos):", constant_index_countries)
else:
    print("\nNo hay países con índice constante a lo largo del tiempo (excluyendo años sin datos).")

# Pregunta e): ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?
countries_all_zeros = pivot_table_df[pivot_table_df.sum(axis=1) == 0].index.tolist()

if countries_all_zeros:
    print("\nPaíses con todos los valores de índice a 0 (sin datos reales):", countries_all_zeros)
    print("Esto se debe a que estos países no tenían registros de 'indice' en el DataFrame original `df` para los años cubiertos, o todos sus registros de índice eran NaN. La función `pivot_table` con `fill_value=0` asignó 0 a estos años sin datos.")
else:
    print("\nNo hay países sin datos de índice (es decir, ningún país con todos los valores a 0).")

anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Afghanistán,35.5,40.17,28.25,39.17,44.25,56.50,59.25,54.25,51.67,37.36,37.07,37.44,37.75,39.46,37.28,36.55
Albania,0.0,6.50,11.50,14.17,18.00,25.50,16.00,21.75,21.50,30.88,29.92,28.77,29.92,29.92,29.49,29.84
Alemania,1.5,1.33,2.00,4.00,5.50,5.75,4.50,3.50,4.25,10.24,10.23,11.47,14.80,14.97,14.39,14.60
Algeria,31.0,33.00,43.50,40.33,40.00,40.50,31.33,49.56,47.33,36.54,36.26,36.63,41.69,42.83,43.13,45.75
Andorra,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,6.82,6.82,19.87,19.87,21.03,22.21,24.63



País con el mayor valor de índice global: Kosovo (Año: 2014, Índice: 64536.0)
País con el menor valor de índice (distinto de cero) global: Austria (Año: 2009, Índice: 0.5)

Año con el promedio de índice más alto: 2013 (Promedio: 449.11)
Año con el promedio de índice más bajo: 2001 (Promedio: 20.03)

País con mayor variabilidad en el índice: Kosovo (Variabilidad: 46098.00)

Países con índice constante a lo largo del tiempo (excluyendo años sin datos): ['Antigua y Barbuda', 'Granada']

No hay países sin datos de índice (es decir, ningún país con todos los valores a 0).
